In [2]:
!pip install -q transformers

In [3]:
!pip install -q datasets

In [11]:
!pip install -q transformers datasets accelerate

In [12]:
!pip install -q seqeval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [6]:
!pip install -q accelerate

In [2]:
import torch

print("=" * 50)
print("GPU CHECK")
print("=" * 50)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: GPU is not available!")

GPU CHECK
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from datasets import load_from_disk

DATA_PATH = "/content/drive/MyDrive/clinic_ai/vietmed_ner"

dataset = load_from_disk(DATA_PATH)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['words', 'tags', 'labels', 'text'],
        num_rows: 4616
    })
    validation: Dataset({
        features: ['words', 'tags', 'labels', 'text'],
        num_rows: 1154
    })
    test: Dataset({
        features: ['words', 'tags', 'labels', 'text'],
        num_rows: 3497
    })
})


In [5]:
print("Train:", len(dataset["train"]))
print("Validation:", len(dataset["validation"]))
print("Test:", len(dataset["test"]))

Train: 4616
Validation: 1154
Test: 3497


In [6]:
train_dataset = dataset["train"]
validation_dataset = dataset["validation"]
test_dataset = dataset["test"]

print(train_dataset.features)

{'words': List(Value('string')), 'tags': List(Value('int64')), 'labels': List(Value('string')), 'text': Value('string')}


In [7]:
sample = train_dataset[0]

print("WORDS:")
print(sample["words"])

print("\nTAGS:")
print(sample["tags"])

print("\nLABELS:")
print(sample["labels"])

print("\nTEXT:")
print(sample["text"])

WORDS:
['thì', 'cũng', 'giống', 'như', 'ba', 'má', 'mình', 'đã', 'từng', 'bị', 'béo', 'phì', 'rồi', 'bị', 'bệnh', 'này', 'bệnh', 'kia', 'những', 'người', 'thân', 'quen', 'của', 'mình', 'mình', 'biết', 'bị']

TAGS:
[22, 22, 22, 22, 13, 13, 22, 22, 22, 22, 2, 8, 22, 2, 8, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22]

LABELS:
['0', '0', '0', '0', 'B-GENDER', 'B-GENDER', '0', '0', '0', '0', 'B-DISEASESYMTOM', 'I-DISEASESYMTOM', '0', 'B-DISEASESYMTOM', 'I-DISEASESYMTOM', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0']

TEXT:
thì cũng giống như ba má mình đã từng bị béo phì rồi bị bệnh này bệnh kia những người thân quen của mình mình biết bị


In [8]:
all_labels = set()

for split in [
    train_dataset,
    validation_dataset,
    test_dataset
]:
    for labels in split["labels"]:
        all_labels.update(labels)

label_names = sorted(all_labels)

print("Number of labels:", len(label_names))
print(label_names)

Number of labels: 37
['0', 'B-AGE', 'B-DATETIME', 'B-DIAGNOSTICS', 'B-DISEASESYMTOM', 'B-DRUGCHEMICAL', 'B-FOODDRINK', 'B-GENDER', 'B-LOCATION', 'B-MEDDEVICETECHNIQUE', 'B-OCCUPATION', 'B-ORGAN', 'B-ORGANIZATION', 'B-PERSONALCARE', 'B-PREVENTIVEMED', 'B-SURGERY', 'B-TRANSPORTATION', 'B-TREATMENT', 'B-UNITCALIBRATOR', 'I-AGE', 'I-DATETIME', 'I-DIAGNOSTICS', 'I-DISEASESYMTOM', 'I-DRUGCHEMICAL', 'I-FOODDRINK', 'I-GENDER', 'I-LOCATION', 'I-MEDDEVICETECHNIQUE', 'I-OCCUPATION', 'I-ORGAN', 'I-ORGANIZATION', 'I-PERSONALCARE', 'I-PREVENTIVEMED', 'I-SURGERY', 'I-TRANSPORTATION', 'I-TREATMENT', 'I-UNITCALIBRATOR']


In [9]:
label2id = {
    label: i
    for i, label in enumerate(label_names)
}

id2label = {
    i: label
    for i, label in enumerate(label_names)
}

print(label2id)
print(id2label)

{'0': 0, 'B-AGE': 1, 'B-DATETIME': 2, 'B-DIAGNOSTICS': 3, 'B-DISEASESYMTOM': 4, 'B-DRUGCHEMICAL': 5, 'B-FOODDRINK': 6, 'B-GENDER': 7, 'B-LOCATION': 8, 'B-MEDDEVICETECHNIQUE': 9, 'B-OCCUPATION': 10, 'B-ORGAN': 11, 'B-ORGANIZATION': 12, 'B-PERSONALCARE': 13, 'B-PREVENTIVEMED': 14, 'B-SURGERY': 15, 'B-TRANSPORTATION': 16, 'B-TREATMENT': 17, 'B-UNITCALIBRATOR': 18, 'I-AGE': 19, 'I-DATETIME': 20, 'I-DIAGNOSTICS': 21, 'I-DISEASESYMTOM': 22, 'I-DRUGCHEMICAL': 23, 'I-FOODDRINK': 24, 'I-GENDER': 25, 'I-LOCATION': 26, 'I-MEDDEVICETECHNIQUE': 27, 'I-OCCUPATION': 28, 'I-ORGAN': 29, 'I-ORGANIZATION': 30, 'I-PERSONALCARE': 31, 'I-PREVENTIVEMED': 32, 'I-SURGERY': 33, 'I-TRANSPORTATION': 34, 'I-TREATMENT': 35, 'I-UNITCALIBRATOR': 36}
{0: '0', 1: 'B-AGE', 2: 'B-DATETIME', 3: 'B-DIAGNOSTICS', 4: 'B-DISEASESYMTOM', 5: 'B-DRUGCHEMICAL', 6: 'B-FOODDRINK', 7: 'B-GENDER', 8: 'B-LOCATION', 9: 'B-MEDDEVICETECHNIQUE', 10: 'B-OCCUPATION', 11: 'B-ORGAN', 12: 'B-ORGANIZATION', 13: 'B-PERSONALCARE', 14: 'B-PREVENTI

In [11]:
from transformers import AutoTokenizer

MODEL_NAME = "vinai/phobert-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
)

print("Tokenizer:", type(tokenizer))
print("Fast:", tokenizer.is_fast)
print("CLS:", tokenizer.cls_token)
print("SEP:", tokenizer.sep_token)
print("PAD:", tokenizer.pad_token)

Tokenizer: <class 'transformers.models.phobert.tokenization_phobert.PhobertTokenizer'>
Fast: False
CLS: <s>
SEP: </s>
PAD: <pad>


In [12]:
MAX_LENGTH = 256


def tokenize_function(example):
    input_ids = []
    attention_mask = []
    aligned_labels = []

    words = example["words"]
    labels = example["labels"]

    for word, label in zip(words, labels):

        tokens = tokenizer(
            word,
            add_special_tokens=False,
        )

        word_input_ids = tokens["input_ids"]

        if not word_input_ids:
            continue

        # Token đầu tiên nhận label
        input_ids.append(word_input_ids[0])
        attention_mask.append(1)
        aligned_labels.append(label2id[label])

        # Các sub-token còn lại bỏ qua loss
        for token_id in word_input_ids[1:]:
            input_ids.append(token_id)
            attention_mask.append(1)
            aligned_labels.append(-100)

    # CLS
    input_ids = (
        [tokenizer.cls_token_id]
        + input_ids
        + [tokenizer.sep_token_id]
    )

    attention_mask = (
        [1]
        + attention_mask
        + [1]
    )

    aligned_labels = (
        [-100]
        + aligned_labels
        + [-100]
    )

    # Truncate
    input_ids = input_ids[:MAX_LENGTH]
    attention_mask = attention_mask[:MAX_LENGTH]
    aligned_labels = aligned_labels[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": aligned_labels,
    }

In [13]:
train_tokenized = train_dataset.map(
    tokenize_function,
    remove_columns=train_dataset.column_names,
)

validation_tokenized = validation_dataset.map(
    tokenize_function,
    remove_columns=validation_dataset.column_names,
)

test_tokenized = test_dataset.map(
    tokenize_function,
    remove_columns=test_dataset.column_names,
)

In [14]:
sample = train_tokenized[0]

print(sample)

{'labels': [-100, 0, 0, 0, 0, 7, 7, 0, 0, 0, 0, 4, 22, 0, 4, 22, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100], 'input_ids': [0, 54, 32, 475, 42, 346, 5001, 68, 14, 150, 45, 2733, 5748, 182, 45, 326, 23, 326, 1357, 21, 18, 1182, 1895, 7, 68, 68, 55, 45, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [15]:
print(
    "input_ids:",
    len(sample["input_ids"])
)

print(
    "attention_mask:",
    len(sample["attention_mask"])
)

print(
    "labels:",
    len(sample["labels"])
)

input_ids: 29
attention_mask: 29
labels: 29


In [16]:


from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
)

MODEL_NAME = "vinai/phobert-base"

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  543MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [19]:
print("Number of labels:", len(label_names))
print("Model loaded successfully")

Number of labels: 37
Model loaded successfully


In [20]:
print("Model device:", model.device)

if torch.cuda.is_available():
    model = model.cuda()
    print("Model moved to:", model.device)

Model device: cpu
Model moved to: cuda:0


In [21]:
from transformers import TrainingArguments

OUTPUT_DIR = "/content/drive/MyDrive/clinic_ai/symptom_ner_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    weight_decay=0.01,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=5,

    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,

    report_to="none",

    fp16=torch.cuda.is_available(),
)

In [22]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=validation_tokenized,

    data_collator=data_collator,

    processing_class=tokenizer,
)

In [23]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.673843,0.527852
2,0.407576,0.359065
3,0.312299,0.306229
4,0.256606,0.278068
5,0.235901,0.265832


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1445, training_loss=0.4434984603142656, metrics={'train_runtime': 250.3306, 'train_samples_per_second': 92.198, 'train_steps_per_second': 5.772, 'total_flos': 402802052663232.0, 'train_loss': 0.4434984603142656, 'epoch': 5.0})

In [24]:
eval_result = trainer.evaluate()

print(eval_result)

Training Loss,Validation Loss,Epoch
0.235901,0.265832,5


{'eval_loss': 0.26583242416381836}


In [25]:
FINAL_MODEL_DIR = (
    "/content/drive/MyDrive/"
    "clinic_ai/symptom_ner_model/final"
)

trainer.save_model(FINAL_MODEL_DIR)

tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)

print("Model saved to:")
print(FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to:
/content/drive/MyDrive/clinic_ai/symptom_ner_model/final


In [26]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
)

tokenizer_test = AutoTokenizer.from_pretrained(
    FINAL_MODEL_DIR,
    use_fast=False,
)

model_test = AutoModelForTokenClassification.from_pretrained(
    FINAL_MODEL_DIR
)

print("Model loaded successfully")
print("Number of labels:", model_test.config.num_labels)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded successfully
Number of labels: 37


In [38]:
text = "Tôi không bị đau đầu và chóng mặt trong 3 ngày nay"
inputs = tokenizer_test(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=256,
)

if torch.cuda.is_available():
    inputs = {
        key: value.cuda()
        for key, value in inputs.items()
    }
    model_test = model_test.cuda()

with torch.no_grad():
    outputs = model_test(**inputs)

predictions = torch.argmax(
    outputs.logits,
    dim=-1,
)

print(predictions)

tensor([[ 4,  0,  0,  0,  4, 22,  0,  4, 22,  0,  2, 20, 20,  4]],
       device='cuda:0')


In [39]:
predicted_labels = [
    model_test.config.id2label[int(pred)]
    for pred in predictions[0]
]

tokens = tokenizer_test.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

for token, label in zip(tokens, predicted_labels):
    print(f"{token:20} -> {label}")

<s>                  -> B-DISEASESYMTOM
Tôi                  -> 0
không                -> 0
bị                   -> 0
đau                  -> B-DISEASESYMTOM
đầu                  -> I-DISEASESYMTOM
và                   -> 0
chóng                -> B-DISEASESYMTOM
mặt                  -> I-DISEASESYMTOM
trong                -> 0
3                    -> B-DATETIME
ngày                 -> I-DATETIME
nay                  -> I-DATETIME
</s>                 -> B-DISEASESYMTOM


In [32]:
entities = []

current_tokens = []

for token, label in zip(tokens, predicted_labels):

    if label.startswith("B-SYMPTOM"):
        if current_tokens:
            entities.append(
                " ".join(current_tokens)
            )

        current_tokens = [token]

    elif label.startswith("I-SYMPTOM"):
        current_tokens.append(token)

    else:
        if current_tokens:
            entities.append(
                " ".join(current_tokens)
            )
            current_tokens = []

if current_tokens:
    entities.append(
        " ".join(current_tokens)
    )

print(entities)

[]


In [34]:
tokens = tokenizer_test.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

predicted_ids = predictions[0].tolist()

predicted_labels = [
    model_test.config.id2label[int(pred_id)]
    for pred_id in predicted_ids
]

for token, label in zip(tokens, predicted_labels):
    print(f"{token:20} -> {label}")

<s>                  -> B-DISEASESYMTOM
Tôi                  -> 0
bị                   -> 0
đau                  -> B-DISEASESYMTOM
đầu                  -> I-DISEASESYMTOM
và                   -> 0
chóng                -> B-DISEASESYMTOM
mặt                  -> I-DISEASESYMTOM
trong                -> 0
3                    -> B-DATETIME
ngày                 -> I-DATETIME
nay                  -> I-DATETIME
</s>                 -> B-DISEASESYMTOM


In [35]:
for token, pred_id, label in zip(
    tokens,
    predicted_ids,
    predicted_labels
):
    print(
        f"{token:20} "
        f"ID={pred_id:<3} "
        f"LABEL={label}"
    )

<s>                  ID=4   LABEL=B-DISEASESYMTOM
Tôi                  ID=0   LABEL=0
bị                   ID=0   LABEL=0
đau                  ID=4   LABEL=B-DISEASESYMTOM
đầu                  ID=22  LABEL=I-DISEASESYMTOM
và                   ID=0   LABEL=0
chóng                ID=4   LABEL=B-DISEASESYMTOM
mặt                  ID=22  LABEL=I-DISEASESYMTOM
trong                ID=0   LABEL=0
3                    ID=2   LABEL=B-DATETIME
ngày                 ID=20  LABEL=I-DATETIME
nay                  ID=20  LABEL=I-DATETIME
</s>                 ID=4   LABEL=B-DISEASESYMTOM


In [36]:
print(label_names)

['0', 'B-AGE', 'B-DATETIME', 'B-DIAGNOSTICS', 'B-DISEASESYMTOM', 'B-DRUGCHEMICAL', 'B-FOODDRINK', 'B-GENDER', 'B-LOCATION', 'B-MEDDEVICETECHNIQUE', 'B-OCCUPATION', 'B-ORGAN', 'B-ORGANIZATION', 'B-PERSONALCARE', 'B-PREVENTIVEMED', 'B-SURGERY', 'B-TRANSPORTATION', 'B-TREATMENT', 'B-UNITCALIBRATOR', 'I-AGE', 'I-DATETIME', 'I-DIAGNOSTICS', 'I-DISEASESYMTOM', 'I-DRUGCHEMICAL', 'I-FOODDRINK', 'I-GENDER', 'I-LOCATION', 'I-MEDDEVICETECHNIQUE', 'I-OCCUPATION', 'I-ORGAN', 'I-ORGANIZATION', 'I-PERSONALCARE', 'I-PREVENTIVEMED', 'I-SURGERY', 'I-TRANSPORTATION', 'I-TREATMENT', 'I-UNITCALIBRATOR']


In [37]:
for i, label in id2label.items():
    print(i, "->", label)

0 -> 0
1 -> B-AGE
2 -> B-DATETIME
3 -> B-DIAGNOSTICS
4 -> B-DISEASESYMTOM
5 -> B-DRUGCHEMICAL
6 -> B-FOODDRINK
7 -> B-GENDER
8 -> B-LOCATION
9 -> B-MEDDEVICETECHNIQUE
10 -> B-OCCUPATION
11 -> B-ORGAN
12 -> B-ORGANIZATION
13 -> B-PERSONALCARE
14 -> B-PREVENTIVEMED
15 -> B-SURGERY
16 -> B-TRANSPORTATION
17 -> B-TREATMENT
18 -> B-UNITCALIBRATOR
19 -> I-AGE
20 -> I-DATETIME
21 -> I-DIAGNOSTICS
22 -> I-DISEASESYMTOM
23 -> I-DRUGCHEMICAL
24 -> I-FOODDRINK
25 -> I-GENDER
26 -> I-LOCATION
27 -> I-MEDDEVICETECHNIQUE
28 -> I-OCCUPATION
29 -> I-ORGAN
30 -> I-ORGANIZATION
31 -> I-PERSONALCARE
32 -> I-PREVENTIVEMED
33 -> I-SURGERY
34 -> I-TRANSPORTATION
35 -> I-TREATMENT
36 -> I-UNITCALIBRATOR
